# 13A — Suspicious VV/VH scenes: fixed-point radiometric QA

**Priority 1.** Compare exactly the same geographic sample points across acquisitions. Test whether VV histograms, VH concentrations, incidence angles or threshold fractions shift between suspicious scenes. Selection of target dates was exploratory, not prespecified.

**Execution:** This is a source workflow, NOT results. A successful extract needs your own data / Earth Engine session. Do not treat low-VV detections as validated water, flood depth or methane flux.

In [1]:
# Every notebook uses the same ROOT; no hard-coded drive or EE project ID.
from pathlib import Path
import os, sys
starting = Path(os.environ.get("HAKALUKI_ROOT", Path.cwd())).expanduser().resolve()
for candidate in [starting, *starting.parents]:
    if (candidate/"src/analysis13/core.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Extract the ZIP into the project root and open Jupyter there, or set HAKALUKI_ROOT")
sys.path.insert(0, str(ROOT))
print("Research root:", ROOT)


Research root: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project


## Local file checks and offline baseline

Runs only on the completed Notebook 12 CSV, no Earth Engine required. Output goes to `results/tables/analysis13` and `results/figures/analysis13`.

In [2]:
from src.analysis13.core import preflight
from src.analysis13.offline import run_offline
from src.analysis13.pairs import PAIRS
print(preflight(ROOT))
print(run_offline(ROOT))
for pair in PAIRS:
    print(pair.name, pair.before_local, pair.after_local, pair.orbit_pass, pair.reason)

{'root': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project', 'scene_csv': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\results\\tables\\scene12_S1_per_acquisition_2022_2025.csv', 'boundary': 'F:\\Wetland Test\\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\\data\\boundary\\hakaluki_boundary.gpkg', 'ee': None}
{'created_utc': '2026-10-14T17:05:54.844148+00:00', 'status': 'exploratory_reanalysis_of_uploaded_scene12_outputs_not_earth_engine_rerun', 'n_scenes': 94, 'n_transitions': 86, 'n_same_sign': 66, 'n_strong_same_sign_20km2': 12, 'years': [2022, 2023, 2024, 2025], 'max_threshold_span_km2': 133.19901915697625, 'all_numerics_from': 'scene12_S1_per_acquisition_2022_2025.csv', 'limitations': 'Area screening is unvalidated; acquisition differences are not directly water change; fixed VV thresholds and orbit effects remain.'}
2022_may_expansion 2022-05-11 2022-05-23 ASCENDING large May low-VV increase
2022_june_stable 2022-06-04 2022-06-28 ASCENDING al

## Fixed-coordinate Earth Engine VV/VH samples

Configure EE_PROJECT in your local environment. `13A` samples 550 fixed spatial points per scene; it can take several minutes. No physical water labels are inferred.

In [3]:
from src.analysis13.radiometry import run_radiometry
RUN_13A_EARTH_ENGINE = True
if RUN_13A_EARTH_ENGINE:
    qa = run_radiometry(ROOT, n_points=550)
    print(qa)
else:
    print("13A EE radiometry deferred; offline QC already completed.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Sampling scene 1/11: S1A_IW_GRDH_1SDV_20220511T115642_20220511T11
Sampling scene 2/11: S1A_IW_GRDH_1SDV_20220523T115643_20220523T11
Sampling scene 3/11: S1A_IW_GRDH_1SDV_20220604T115644_20220604T11
Sampling scene 4/11: S1A_IW_GRDH_1SDV_20220628T115646_20220628T11
Sampling scene 5/11: S1A_IW_GRDH_1SDV_20230611T115649_20230611T11
Sampling scene 6/11: S1A_IW_GRDH_1SDV_20230623T115650_20230623T11
Sampling scene 7/11: S1A_IW_GRDH_1SDV_20240605T115652_20240605T11
Sampling scene 8/11: S1A_IW_GRDH_1SDV_20240617T115651_20240617T11
Sampling scene 9/11: S1A_IW_GRDH_1SDV_20240629T115651_20240629T11
Sampling scene 10/11: S1A_IW_GRDH_1SDV_20250521T234742_20250521T23
Sampling scene 11/11: S1A_IW_GRDH_1SDV_20250602T234742_20250602T23
{'created_utc': '2026-10-14T17:06:32.153139+00:00', 'status': 'executed_radiometric_screening_not_validation', 'total_fixed_points': 550, 'scene_count': 11, 'pair_count': 6, 'same_coordinate_points': True, 'warning': 'Threshold changes and backscatter shifts cannot alone 

## Inspect the generated diagnostics

Review the CSVs, especially VH modal-bin fraction, matched ΔVV/ΔVH, scene support and incidence angle. Differences are not yet proven hydrological.

In [ ]:
from pathlib import Path
for p in sorted((ROOT/'results/tables/analysis13').glob('13A_*.csv')):
    print(p.name, round(p.stat().st_size/1024, 1), 'KiB')
for p in sorted((ROOT/'results/figures/analysis13').glob('13A_*.png')):
    print('Figure:',p)